# Práctica 9 — K-means vs DBScan en Diamantes
**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**

---
**Objetivo:** Segmentar diamantes con K-means y detectar diamantes outlier con DBScan. Comparar ambos enfoques (centroides vs densidad).

**Dataset:** `diamonds` de seaborn (muestra de 1500 diamantes)

⚠️ **Instrucciones:**
- Celdas marcadas con `# 🔧 TU CÓDIGO` debes completarlas.
- Responde las preguntas ❓ en celdas Markdown.
- Guarda una copia en Drive: `Archivo → Guardar una copia en Drive`

## Parte 0 — Setup (Solo ejecutar)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster       import KMeans, DBSCAN
from sklearn.neighbors     import NearestNeighbors
from sklearn.metrics       import silhouette_score

print('✅ Librerías cargadas')

## Parte 1 — Preparar y Escalar

In [ ]:
df = sns.load_dataset('diamonds').sample(1500, random_state=42).reset_index(drop=True)
print(f'Diamantes en la muestra: {df.shape[0]}')
print(df[['carat', 'price', 'x', 'y', 'z']].describe().round(2))

In [ ]:
# 🔧 TU CÓDIGO
# 1. Usa 'carat', 'price', 'x', 'y', 'z'
# 2. Escala con StandardScaler

features = ['carat', 'price', 'x', 'y', 'z']
X = df[features].copy()

scaler   = ___________          # StandardScaler
X_scaled = ___________          # fit_transform sobre X

print('Forma de X_scaled:', X_scaled.shape)
print('Media (~0):', X_scaled.mean(axis=0).round(2))

### ❓ Antes de modelar
*(Responde aquí en Markdown)*

1. ¿Por qué es crítico escalar (compara rangos de carat y price)?
2. ¿Qué grupos de mercado esperarías encontrar?

## Parte 2 — Segmentar con K-means

In [ ]:
# 🔧 TU CÓDIGO
# Para k de 2 a 8: inercia + silhouette_score

print(f"{'k':>3} {'inercia':>12} {'silhouette':>12}")
for k in range(2, 9):
    km  = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    lab = km.fit_predict(X_scaled)
    sil = ___________          # silhouette_score(X_scaled, lab)
    print(f'{k:>3} {km.inertia_:>12.1f} {sil:>12.3f}')

In [ ]:
# 🔧 TU CÓDIGO
# Elige tu mejor k y ajusta K-means; guarda df['cluster_km'] y grafica

mejor_k = ___________          # tu elección, p.ej. 3
km = KMeans(n_clusters=mejor_k, init='k-means++', n_init=10, random_state=42)
df['cluster_km'] = km.fit_predict(X_scaled)

plt.figure(figsize=(7.5, 5))
plt.scatter(df['carat'], df['price'], c=df['cluster_km'], cmap='viridis', s=20, edgecolor='k', linewidth=0.2)
plt.xlabel('Quilates (carat)')
plt.ylabel('Precio (USD)')
plt.title(f'K-means (k={mejor_k}) — Segmentos de diamantes')
plt.colorbar(label='Clúster')
plt.tight_layout()
plt.show()

print(df.groupby('cluster_km')[features].mean().round(1))

### ❓ Sobre los segmentos
*(Responde aquí en Markdown)*

1. ¿Qué k elegiste y por qué?
2. Describe cada segmento en palabras.

## Parte 3 — Elegir eps para DBScan

In [ ]:
# 🔧 TU CÓDIGO
# Gráfico de k-distancias (min_samples=6) para elegir eps

min_samples = 6
nn = NearestNeighbors(n_neighbors=min_samples)
nn.fit(X_scaled)
distancias, _ = nn.kneighbors(X_scaled)
k_dist = np.sort(distancias[:, -1])

plt.figure(figsize=(8, 4.5))
# ___ tu código: plt.plot(k_dist, ...) ___
plt.xlabel('Puntos ordenados')
plt.ylabel(f'Distancia al vecino {min_samples}')
plt.title('Gráfico de k-distancias — busca el codo para eps')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ❓ Sobre eps
*(Responde aquí en Markdown)*

1. ¿A qué altura está el codo? Ese es tu candidato a eps.
2. ¿Qué pasaría con eps mucho más pequeño? ¿Y más grande?

## Parte 4 — DBScan y Detección de Outliers

In [ ]:
# 🔧 TU CÓDIGO
# DBSCAN con tu eps (p.ej. 0.5) y min_samples=6

db = ___________               # DBSCAN(eps=..., min_samples=6)
df['cluster_db'] = db.fit_predict(X_scaled)

n_clusters = len(set(df['cluster_db'])) - (1 if -1 in df['cluster_db'].values else 0)
n_outliers = (df['cluster_db'] == -1).sum()
print(f'Clústeres encontrados: {n_clusters}')
print(f'Outliers detectados:   {n_outliers}')

In [ ]:
# 🔧 TU CÓDIGO
# Grafica carat vs price: clústeres en color, outliers (-1) en X roja

es_outlier = df['cluster_db'] == -1

plt.figure(figsize=(7.5, 5))
# ___ scatter de NO outliers coloreado por cluster_db ___
# ___ scatter de outliers en rojo, marker='x' ___
plt.xlabel('Quilates (carat)')
plt.ylabel('Precio (USD)')
plt.title('DBScan — Diamantes outlier en rojo')
plt.legend()
plt.tight_layout()
plt.show()

print('Algunos diamantes outlier:')
print(df[es_outlier][['carat', 'price', 'x', 'y', 'z']].head(10))

### ❓ Sobre los outliers
*(Responde aquí en Markdown)*

1. ¿Qué tienen de raro los diamantes outlier (precio vs tamaño)?
2. ¿Para qué le serviría esto a una joyería o a un sistema de detección de fraude?

## Parte 5 — Comparativa Final

In [ ]:
# 🔧 TU CÓDIGO
# Dos subplots (carat vs price): izquierda K-means, derecha DBScan

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# ___ izquierda: colorear por df['cluster_km'] ___
axes[0].set_title('K-means')
# ___ derecha: colorear por df['cluster_db'] ___
axes[1].set_title('DBScan (rojo = outlier)')
for ax in axes:
    ax.set_xlabel('carat'); ax.set_ylabel('price')
plt.suptitle('K-means vs DBScan en diamantes', fontsize=12)
plt.tight_layout()
plt.show()

### ❓ Decisión final
*(Responde aquí en Markdown)*

1. Para segmentar el mercado de diamantes, ¿cuál algoritmo prefieres y por qué?
2. Para detectar diamantes anómalos, ¿cuál es claramente mejor?

---
**Conclusión integradora:** ¿Cuándo usar K-means y cuándo DBScan? Menciona al menos 3 criterios (forma de los grupos, número conocido o no de clústeres, detección de outliers, escala/velocidad).